In [1]:
import os
os.environ['HF_ENDPOINT'] = "https://hf-mirror.com"
# os.environ['HF_HOME'] = "/root/autodl-shared/hf_cache"  # autodl
os.environ['HF_HOME'] = "/root/shared-nvme/hf_cache"  # paratera

## (1) Load Model & Weights from HuggingFace

In [2]:
import json
import jax
import jax.numpy as jnp
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download
from safetensors import safe_open
from misc.mamba.model import Mamba, ModelArgs

def load_flax_mamba_hf(model_name: str):
    config_path = hf_hub_download(repo_id=model_name, filename="config.json")
    model_path = hf_hub_download(repo_id=model_name, filename="model.safetensors")
    return load_flax_mamba(config_path, model_path)

def load_flax_mamba(config_path: str, model_path: str):
    # 1. 解析 config.json 获取模型结构
    with open(config_path, 'r') as f:
        config = json.load(f)

    args = ModelArgs(
        d_model=config.get('hidden_size', config.get('d_model', 768)),
        n_layer=config.get('num_hidden_layers', config.get('n_layer', 24)),
        vocab_size=config.get('vocab_size', 50277)
    )

    model = Mamba(args)

    # 2. 加载 safetensors 权重并转换为 Flax 参数字典
    params = {}

    with safe_open(model_path, framework="np", device="cpu") as f:
        # 提取 Embedding (注意 s) 和最后的 Norm
        params['embedding'] = {'embedding': jnp.array(f.get_tensor('backbone.embeddings.weight'))}
        params['norm_f'] = {'weight': jnp.array(f.get_tensor('backbone.norm_f.weight'))}

        # 遍历每一层进行权重映射
        for i in range(args.n_layer):
            layer_name = f'layers_{i}'
            pt_prefix = f'backbone.layers.{i}'

            # 卷积层权重维度转换: PyTorch(C_out, C_in, L) -> Flax(L, C_in, C_out)
            conv_weight = f.get_tensor(f'{pt_prefix}.mixer.conv1d.weight')
            conv_weight = jnp.transpose(conv_weight, (2, 1, 0))

            # 线性层需要转置 (.T)
            params[layer_name] = {
                'norm': {'weight': jnp.array(f.get_tensor(f'{pt_prefix}.norm.weight'))},
                'mixer': {
                    'A_log': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.A_log')),
                    'D': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.D')),
                    'in_proj': {'kernel': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.in_proj.weight')).T},
                    'conv1d': {
                        'kernel': conv_weight,
                        'bias': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.conv1d.bias'))
                    },
                    'x_proj': {'kernel': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.x_proj.weight')).T},
                    'dt_proj': {
                        'kernel': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.dt_proj.weight')).T,
                        'bias': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.dt_proj.bias'))
                    },
                    'out_proj': {'kernel': jnp.array(f.get_tensor(f'{pt_prefix}.mixer.out_proj.weight')).T}
                }
            }

    return model, params

# 加载模型和分词器
pretrained_model_name = 'state-spaces/mamba-130m-hf'
tokenizer = AutoTokenizer.from_pretrained('EleutherAI/gpt-neox-20b')
model, params = load_flax_mamba_hf(pretrained_model_name)

# tokenizer_path="/root/.cache/huggingface/hub/models--EleutherAI--gpt-neox-20b/snapshots/c292233c833e336628618a88a648727eb3dff0a7"
# config_path="/root/.cache/huggingface/hub/models--state-spaces--mamba-130m-hf/snapshots/1e76775f628fbf1350fbe4dbb3d971ba64af25a1/config.json"
# model_path="/root/.cache/huggingface/hub/models--state-spaces--mamba-130m-hf/snapshots/1e76775f628fbf1350fbe4dbb3d971ba64af25a1/model.safetensors"
#
# tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
# model, params = load_flax_mamba(config_path, model_path)

print("预训练模型参数与配置已成功加载并转换！")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


预训练模型参数与配置已成功加载并转换！


## (2) Generate Text

In [3]:
from functools import partial

@partial(jax.jit, static_argnames=('top_k',))
def forward_step(params, input_ids, top_k=40):
    logits = model.apply({'params': params}, input_ids)
    next_token_logits = logits[:, -1, :]
    return next_token_logits

def generate(params, tokenizer, prompt: str, n_tokens_to_gen: int = 50, sample: bool = True, top_k: int = 40, seed: int = 10086, demo: bool = False):
    if demo:
        print(prompt, end="")

    key = jax.random.PRNGKey(seed)
    input_ids = jnp.array(tokenizer(prompt, return_tensors='np').input_ids)

    for _ in range(n_tokens_to_gen):
        next_token_logits = forward_step(params, input_ids, top_k)

        if top_k is not None:

            # 1. 排序，找到最大的k个值以及对应的id
            values, _ = jax.lax.top_k(next_token_logits, k=top_k)

            # 2. 取第k大的值
            kth_values = values[:, -1:]

            # 3. 把概率更小的token全部置0，这样一来softmax后它们的概率就是0，相当于从中前k中取样
            next_token_logits = jnp.where(next_token_logits < kth_values, -1e9, next_token_logits)

        # 4. 归一化，转概率，给后面随机采样用
        probs = jax.nn.softmax(next_token_logits, axis=-1)

        # print(probs)

        if sample:
            key, subkey = jax.random.split(key)
            next_indices = jax.random.categorical(subkey, jnp.log(probs + 1e-9), axis=-1)[:, None]
        else:
            next_indices = jnp.argmax(probs, axis=-1)[:, None]

        input_ids = jnp.concatenate([input_ids, next_indices], axis=1)

        if demo:
            print(tokenizer.decode(next_indices[0]), end="")

    if demo:
        return None
    return tokenizer.decode(input_ids[0].tolist())

def generate_demo(prompt: str, n_tokens_to_gen: int = 10, seed: int = 10086):
    return generate(params, tokenizer, prompt, n_tokens_to_gen, seed=seed, demo=True)

In [4]:
generate_demo('Mamba is the')

Mamba is the smallest of the seven species, with a length typically

In [5]:
generate_demo('The meaning of life is ')

The meaning of life is 
"life as a product of human nature".

In [6]:
generate_demo('def reverse_string(')

def reverse_string(self, data_to_reverse):
	

In [7]:
generate_demo('The meaning of life is ', seed=114514)

The meaning of life is 
"a state of being and the state of